# Loan Financial Analysis - ETL & Snowflake Schema


## 1. Extract Data

In [1]:
import pandas as pd

df = pd.read_csv("Loan.csv")
print(df.shape)
df.head()

(20000, 36)


,ApplicationDate,Age,AnnualIncome,CreditScore,EmploymentStatus,EducationLevel,Experience,LoanAmount,LoanDuration,MaritalStatus,...,MonthlyIncome,UtilityBillsPaymentHistory,JobTenure,NetWorth,BaseInterestRate,InterestRate,MonthlyLoanPayment,TotalDebtToIncomeRatio,LoanApproved,RiskScore
0,2018-01-01,45,39948,617,Employed,Master,22,13152,48,Married,...,3329.000000,0.724972,11,126928,0.199652,0.227590,419.805992,0.181077,0,49.0
1,2018-01-02,38,39709,628,Employed,Associate,15,26045,48,Single,...,3309.083333,0.935132,3,43609,0.207045,0.201077,794.054238,0.389852,0,52.0
2,2018-01-03,47,40724,570,Employed,Bachelor,26,17627,36,Married,...,3393.666667,0.872241,6,5205,0.217627,0.212548,666.406688,0.462157,0,52.0
3,2018-01-04,58,69084,545,Employed,High School,34,37898,96,Single,...,5757.000000,0.896155,5,99452,0.300398,0.300911,1047.506980,0.313098,0,54.0
4,2018-01-05,37,103264,594,Employed,Associate,17,9184,36,Married,...,8605.333333,0.941369,5,227019,0.197184,0.175990,330.179140,0.070210,1,36.0


## 2. Check Nulls and Data Types

In [2]:
print(df.isnull().sum())
print()
print(df.dtypes)

ApplicationDate               0
Age                           0
AnnualIncome                  0
CreditScore                   0
EmploymentStatus              0
EducationLevel                0
Experience                    0
LoanAmount                    0
LoanDuration                  0
MaritalStatus                 0
NumberOfDependents            0
HomeOwnershipStatus           0
MonthlyDebtPayments           0
CreditCardUtilizationRate     0
NumberOfOpenCreditLines       0
NumberOfCreditInquiries       0
DebtToIncomeRatio             0
BankruptcyHistory             0
LoanPurpose                   0
PreviousLoanDefaults          0
PaymentHistory                0
LengthOfCreditHistory         0
SavingsAccountBalance         0
CheckingAccountBalance        0
TotalAssets                   0
TotalLiabilities              0
MonthlyIncome                 0
UtilityBillsPaymentHistory    0
JobTenure                     0
NetWorth                      0
BaseInterestRate              0
Interest

## 3. Transform Date

In [3]:
df['ApplicationDate'] = pd.to_datetime(df['ApplicationDate'])

df['Year']    = df['ApplicationDate'].dt.year
df['Month']   = df['ApplicationDate'].dt.month
df['Quarter'] = df['ApplicationDate'].dt.quarter

df[['ApplicationDate','Year','Month','Quarter']].head()

,ApplicationDate,Year,Month,Quarter
0,2018-01-01,2018,1,1
1,2018-01-02,2018,1,1
2,2018-01-03,2018,1,1
3,2018-01-04,2018,1,1
4,2018-01-05,2018,1,1


## 4. Categories: Credit / Loan Size / Age

In [4]:
def credit_category(score):
    if score >= 750: return 'Excellent'
    elif score >= 700: return 'Good'
    elif score >= 650: return 'Fair'
    else: return 'Poor'

df['CreditCategory'] = df['CreditScore'].apply(credit_category)

def loan_size(amount):
    if amount < 10000: return 'Small'
    elif amount < 30000: return 'Medium'
    else: return 'Large'

df['LoanSizeCategory'] = df['LoanAmount'].apply(loan_size)

def age_group(age):
    if age < 30: return 'Young'
    elif age < 45: return 'Middle'
    elif age < 60: return 'Senior'
    else: return 'Elder'

df['AgeGroup'] = df['Age'].apply(age_group)

df[['CreditScore','CreditCategory','LoanAmount','LoanSizeCategory','Age','AgeGroup']].head()

,CreditScore,CreditCategory,LoanAmount,LoanSizeCategory,Age,AgeGroup
0,617,Poor,13152,Medium,45,Senior
1,628,Poor,26045,Medium,38,Middle
2,570,Poor,17627,Medium,47,Senior
3,545,Poor,37898,Large,58,Senior
4,594,Poor,9184,Small,37,Middle


## 5. Risk Level / Default Probability

In [5]:
def risk_level(row):
    score = 0
    if row['CreditScore'] < 600: score += 3
    elif row['CreditScore'] < 700: score += 1
    if row['RiskScore'] > 60: score += 3
    elif row['RiskScore'] > 30: score += 1
    if row['DebtToIncomeRatio'] > 0.4: score += 2
    elif row['DebtToIncomeRatio'] > 0.2: score += 1
    if row['PreviousLoanDefaults'] > 0: score += 3
    if score >= 7: return 'High Risk'
    elif score >= 3: return 'Medium Risk'
    else: return 'Low Risk'

df['RiskLevel_Final'] = df.apply(risk_level, axis=1)

def default_probability_v2(row):
    prob = 0.05
    credit_factor = max(0, (700 - row['CreditScore']) / 400) * 0.4
    default_factor = min(row['PreviousLoanDefaults'], 3) * 0.15
    dti_factor = min(row['DebtToIncomeRatio'], 1) * 0.25
    risk_factor = min(row['RiskScore'] / 100, 1) * 0.2
    total = prob + credit_factor + default_factor + dti_factor + risk_factor
    return round(min(total, 1), 3)

df['DefaultProbability'] = df.apply(default_probability_v2, axis=1)

df['RiskLevel_Final'].value_counts()

RiskLevel_Final
Medium Risk    15642
High Risk       2352
Low Risk        2006
Name: count, dtype: int64

## 6. Financial Strength

In [6]:
def financial_strength(row):
    score = 0
    if row['NetWorth'] > 0: score += 2
    if row['SavingsAccountBalance'] > 10000: score += 2
    if row['DebtToIncomeRatio'] < 0.3: score += 2
    if row['CreditScore'] > 700: score += 2
    if score >= 6: return 'Strong'
    elif score >= 3: return 'Moderate'
    else: return 'Weak'

df['FinancialStrength'] = df.apply(financial_strength, axis=1)
df['FinancialStrength'].value_counts()

FinancialStrength
Moderate    11194
Weak         7491
Strong       1315
Name: count, dtype: int64

## 7. Revenue / Expenses / Profit
Revenue = الفايدة الفعلية على القرض (TotalPayment - LoanAmount).
Expenses = افتراض 30% من الـ Revenue كتكلفة تشغيلية.

In [7]:
df['TotalPayment'] = df['MonthlyLoanPayment'] * df['LoanDuration']
df['Revenue'] = df['TotalPayment'] - df['LoanAmount']
df['Expenses'] = df['Revenue'] * 0.30
df['Profit'] = df['Revenue'] - df['Expenses']

print("Financial Summary")
print("=" * 40)
print(f"Total Revenue:   ${df['Revenue'].sum():,.0f}")
print(f"Total Expenses:  ${df['Expenses'].sum():,.0f}")
print(f"Total Profit:    ${df['Profit'].sum():,.0f}")
print(f"Profit Margin:   {(df['Profit'].sum() / df['Revenue'].sum())*100:.1f}%")

Financial Summary
Total Revenue:   $359,642,776
Total Expenses:  $107,892,833
Total Profit:    $251,749,943
Profit Margin:   70.0%


## 8. Customer Segmentation

In [8]:
def income_level(income):
    if income < 40000: return 'Low Income'
    elif income < 80000: return 'Middle Income'
    else: return 'High Income'

df['IncomeLevel'] = df['AnnualIncome'].apply(income_level)

def customer_segment(row):
    age_tag = 'Young' if row['Age'] < 40 else 'Old'
    risk_tag = 'Risky' if row['RiskLevel_Final'] == 'High Risk' else 'Safe'
    return f"{age_tag} - {risk_tag}"

df['CustomerSegment'] = df.apply(customer_segment, axis=1)

print(df['IncomeLevel'].value_counts())
print()
print(df['CustomerSegment'].value_counts())

IncomeLevel
Middle Income    8074
Low Income       7627
High Income      4299
Name: count, dtype: int64

CustomerSegment
Old - Safe       9041
Young - Safe     8607
Young - Risky    1333
Old - Risky      1019
Name: count, dtype: int64


## 9. KPIs

In [9]:
kpis = pd.DataFrame({
    "KPI": [
        "Total Revenue", "Total Profit", "Average Loan Amount",
        "Approval Rate (%)", "Default Rate (%)",
        "Average Risk Score", "Average Interest Rate"
    ],
    "Value": [
        df['Revenue'].sum(),
        df['Profit'].sum(),
        df['LoanAmount'].mean(),
        df['LoanApproved'].mean() * 100,
        df['PreviousLoanDefaults'].mean() * 100,
        df['RiskScore'].mean(),
        df['InterestRate'].mean()
    ]
})
kpis

,KPI,Value
0,Total Revenue,3.596428e+08
1,Total Profit,2.517499e+08
2,Average Loan Amount,2.488287e+04
3,Approval Rate (%),2.390000e+01
4,Default Rate (%),1.000500e+01
5,Average Risk Score,5.076678e+01
6,Average Interest Rate,2.391100e-01


## 10. Dimension Tables (Snowflake Schema)

In [10]:
# DIM_Date
dim_date = df[['ApplicationDate','Year','Month','Quarter']].drop_duplicates().reset_index(drop=True)
dim_date.insert(0, 'DateID', range(1, len(dim_date) + 1))

# DIM_LoanPurpose
dim_purpose = df[['LoanPurpose']].drop_duplicates().reset_index(drop=True)
dim_purpose.insert(0, 'PurposeID', range(1, len(dim_purpose) + 1))

# DIM_CreditProfile
dim_credit = df[['CreditScore','CreditCategory','CreditCardUtilizationRate',
                  'NumberOfOpenCreditLines','NumberOfCreditInquiries',
                  'LengthOfCreditHistory','PaymentHistory',
                  'BankruptcyHistory','PreviousLoanDefaults']].drop_duplicates().reset_index(drop=True)
dim_credit.insert(0, 'CreditID', range(1, len(dim_credit) + 1))

# DIM_Risk
dim_risk = df[['RiskScore','RiskLevel_Final','DefaultProbability','FinancialStrength']].drop_duplicates().reset_index(drop=True)
dim_risk.insert(0, 'RiskID', range(1, len(dim_risk) + 1))

# DIM_Customer
dim_customer = df[['Age','AgeGroup','AnnualIncome','IncomeLevel','MonthlyIncome',
                    'EmploymentStatus','EducationLevel','MaritalStatus',
                    'NumberOfDependents','HomeOwnershipStatus','JobTenure',
                    'CustomerSegment']].drop_duplicates().reset_index(drop=True)
dim_customer.insert(0, 'CustomerID', range(1, len(dim_customer) + 1))

print(f"DIM_Date: {len(dim_date):,}")
print(f"DIM_LoanPurpose: {len(dim_purpose):,}")
print(f"DIM_CreditProfile: {len(dim_credit):,}")
print(f"DIM_Risk: {len(dim_risk):,}")
print(f"DIM_Customer: {len(dim_customer):,}")

DIM_Date: 20,000
DIM_LoanPurpose: 5
DIM_CreditProfile: 20,000
DIM_Risk: 11,259
DIM_Customer: 19,999


## 11. (Foreign Keys)

In [11]:
df = df.merge(dim_date[['DateID','ApplicationDate']], on='ApplicationDate', how='left')
df = df.merge(dim_purpose[['PurposeID','LoanPurpose']], on='LoanPurpose', how='left')

df = df.merge(
    dim_customer[['CustomerID','Age','AgeGroup','AnnualIncome','IncomeLevel','MonthlyIncome',
                  'EmploymentStatus','EducationLevel','MaritalStatus',
                  'NumberOfDependents','HomeOwnershipStatus','JobTenure','CustomerSegment']],
    on=['Age','AgeGroup','AnnualIncome','IncomeLevel','MonthlyIncome','EmploymentStatus',
        'EducationLevel','MaritalStatus','NumberOfDependents','HomeOwnershipStatus',
        'JobTenure','CustomerSegment'],
    how='left'
)

df = df.merge(
    dim_credit[['CreditID','CreditScore','CreditCategory','CreditCardUtilizationRate',
                'NumberOfOpenCreditLines','NumberOfCreditInquiries',
                'LengthOfCreditHistory','PaymentHistory','BankruptcyHistory','PreviousLoanDefaults']],
    on=['CreditScore','CreditCategory','CreditCardUtilizationRate','NumberOfOpenCreditLines',
        'NumberOfCreditInquiries','LengthOfCreditHistory','PaymentHistory',
        'BankruptcyHistory','PreviousLoanDefaults'],
    how='left'
)

df = df.merge(
    dim_risk[['RiskID','RiskScore','RiskLevel_Final','DefaultProbability','FinancialStrength']],
    on=['RiskScore','RiskLevel_Final','DefaultProbability','FinancialStrength'],
    how='left'
)

# Add LoanID + remove any duplicate matches from merges
df.insert(0, 'LoanID', range(1, len(df) + 1))
print("Duplicate LoanIDs before cleanup:", df['LoanID'].duplicated().sum())
df = df.drop_duplicates(subset='LoanID', keep='first').reset_index(drop=True)

print(f"Final df shape: {df.shape}")
df[['LoanID','DateID','PurposeID','CustomerID','CreditID','RiskID']].head()

Duplicate LoanIDs before cleanup: 0
Final df shape: (20000, 57)


,LoanID,DateID,PurposeID,CustomerID,CreditID,RiskID
0,1,1,1,1,1,1
1,2,2,2,2,2,2
2,3,3,3,3,3,3
3,4,4,1,4,4,4
4,5,5,2,5,5,5


## 12. Build Fact Table

In [12]:
fact_loans = df[[
    'LoanID', 'DateID', 'CustomerID', 'PurposeID', 'CreditID', 'RiskID',
    'LoanAmount', 'LoanDuration', 'InterestRate', 'MonthlyLoanPayment',
    'DebtToIncomeRatio', 'TotalDebtToIncomeRatio',
    'TotalAssets', 'TotalLiabilities', 'NetWorth',
    'SavingsAccountBalance', 'CheckingAccountBalance',
    'Revenue', 'Expenses', 'Profit', 'LoanApproved'
]]

print(f"Fact_Loans: {len(fact_loans):,} rows, {len(fact_loans.columns)} columns")
fact_loans.head()

Fact_Loans: 20,000 rows, 21 columns


,LoanID,DateID,CustomerID,PurposeID,CreditID,RiskID,LoanAmount,LoanDuration,InterestRate,MonthlyLoanPayment,...,TotalDebtToIncomeRatio,TotalAssets,TotalLiabilities,NetWorth,SavingsAccountBalance,CheckingAccountBalance,Revenue,Expenses,Profit,LoanApproved
0,1,1,1,1,1,1,13152,48,0.227590,419.805992,...,0.181077,146111,19183,126928,7632,1202,6998.687595,2099.606278,4899.081316,0
1,2,2,2,2,2,2,26045,48,0.201077,794.054238,...,0.389852,53204,9595,43609,4627,3460,12069.603435,3620.881030,8448.722404,0
2,3,3,3,3,3,3,17627,36,0.212548,666.406688,...,0.462157,25176,128874,5205,886,895,6363.640756,1909.092227,4454.548529,0
3,4,4,4,1,4,4,37898,96,0.300911,1047.506980,...,0.313098,104822,5370,99452,1675,1217,62662.670102,18798.801031,43863.869071,0
4,5,5,5,2,5,5,9184,36,0.175990,330.179140,...,0.070210,244305,17286,227019,1555,4981,2702.449057,810.734717,1891.714340,1


## 13. Save CSV

In [13]:
dim_date.to_csv("Dim_Date.csv", index=False)
dim_customer.to_csv("Dim_Customer.csv", index=False)
dim_purpose.to_csv("Dim_LoanPurpose.csv", index=False)
dim_credit.to_csv("Dim_CreditProfile.csv", index=False)
dim_risk.to_csv("Dim_Risk.csv", index=False)
fact_loans.to_csv("Fact_Loans.csv", index=False)

print("CSV files saved.")

CSV files saved.


## 14. Connect to SQL Server

In [14]:
from sqlalchemy import create_engine

SERVER = "MOAZ\\SQLEXPRESS"
DATABASE = "LoanDW"

engine = create_engine(
    f"mssql+pyodbc://{SERVER}/{DATABASE}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)

with engine.connect() as conn:
    print("Connected successfully!")

Connected successfully!


## 15. Upload Tables on SQL Server 

In [15]:
tables = {
    'Dim_Date': dim_date,
    'Dim_Customer': dim_customer,
    'Dim_LoanPurpose': dim_purpose,
    'Dim_CreditProfile': dim_credit,
    'Dim_Risk': dim_risk,
    'Fact_Loans': fact_loans,
}

for table_name, df_table in tables.items():
    df_table.to_sql(
        table_name,
        engine,
        if_exists='replace',
        index=False,
        chunksize=1000
    )
    print(f"Loaded {table_name}: {len(df_table):,} rows")

print("\nAll tables loaded successfully!")

Loaded Dim_Date: 20,000 rows
Loaded Dim_Customer: 19,999 rows
Loaded Dim_LoanPurpose: 5 rows
Loaded Dim_CreditProfile: 20,000 rows
Loaded Dim_Risk: 11,259 rows
Loaded Fact_Loans: 20,000 rows

All tables loaded successfully!


## 16. Final Check SQL Server

In [16]:
for t in ['Dim_Date','Dim_Customer','Dim_LoanPurpose','Dim_CreditProfile','Dim_Risk','Fact_Loans']:
    cnt = pd.read_sql(f"SELECT COUNT(*) AS cnt FROM {t}", engine)['cnt'][0]
    print(f"{t}: {cnt:,} rows")

Dim_Date: 20,000 rows
Dim_Customer: 19,999 rows
Dim_LoanPurpose: 5 rows
Dim_CreditProfile: 20,000 rows
Dim_Risk: 11,259 rows
Fact_Loans: 20,000 rows


In [17]:
from sqlalchemy import text

# Step 1: Delete data in correct order (Fact first, then Dims)
with engine.begin() as conn:
    conn.execute(text("DELETE FROM Fact_Loans"))
    conn.execute(text("DELETE FROM Dim_Risk"))

print("Old data cleared from Fact_Loans and Dim_Risk")

# Step 2: Re-insert updated data (append, not replace)
dim_risk.to_sql('Dim_Risk', engine, if_exists='append', index=False, chunksize=1000)
fact_loans.to_sql('Fact_Loans', engine, if_exists='append', index=False, chunksize=1000)

print("Dim_Risk and Fact_Loans updated successfully!")

Old data cleared from Fact_Loans and Dim_Risk
Dim_Risk and Fact_Loans updated successfully!
